# ANKA-50M — R3 and M3 Architecture Training

> **Status:** Serious single-mechanism comparison

## Purpose

Define, validate, train, and compare the recurrent Reflective Loop (R3) and sparse Morphing-FFN (M3) architectures under a shared protocol.

## Execution guidance

Run structural, gradient, causal, and VRAM tests before either training arm. Train R3 and M3 to the same token budget before comparing them.

## Primary outputs

R3/M3 checkpoints, validation metrics, throughput, mechanism statistics, and samples.

## Reproducibility notes

- Code-cell outputs and execution counters are intentionally cleared in the public release.
- Seeds, persistent paths, and checkpoint-resume behavior remain in the code.
- Review path constants before execution; historical notebooks may reference earlier run directories.
- Execute sections from top to bottom unless a section explicitly describes an independent experiment arm.


## 1. Configure Colab paths and deterministic execution

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 1 — Colab, paths, and deterministic initialization
from google.colab import drive
drive.mount("/content/drive")

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional
import gc
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Select a GPU runtime for this long experiment.")

DRIVE_RUN = Path("/content/drive/MyDrive/ANKA/run_01")
TOKENIZER_PATH = DRIVE_RUN / "tokenizer/tokenizer.json"
MANIFEST_CANDIDATES = [
    Path("/content/anka_shards/manifest.json"),
    DRIVE_RUN / "shards/manifest.json",
    DRIVE_RUN / "tokenized/manifest.json",
]
MANIFEST_PATH = next((p for p in MANIFEST_CANDIDATES if p.exists()), None)
OUTPUT_ROOT = DRIVE_RUN / "serious_architectures"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("GPU       :", torch.cuda.get_device_name(0))
print("VRAM      :", f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GB")
print("Tokenizer :", TOKENIZER_PATH)
print("Manifest  :", MANIFEST_PATH)
print("Outputlar  :", OUTPUT_ROOT)

assert TOKENIZER_PATH.exists(), f"Tokenizer not found: {TOKENIZER_PATH}"
assert MANIFEST_PATH is not None, "manifest.json not found. MANIFEST_CANDIDATES list fix."

## 2. Define shared model components

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 2 — shared model components
@dataclass
class ANKAConfig:
    name: str
    vocab_size: int = 15_360
    max_seq_len: int = 2_048
    d_model: int = 448
    n_layers: int = 20
    n_heads: int = 7
    n_kv_heads: int = 1
    d_ff: int = 1_216
    rope_theta: float = 10_000.0
    norm_eps: float = 1e-5
    dropout: float = 0.0


@dataclass
class ANKAOutput:
    logits: torch.Tensor
    loss: Optional[torch.Tensor] = None


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        y = x.float()
        y = y * torch.rsqrt(y.square().mean(dim=-1, keepdim=True) + self.eps)
        # Parametreler FP32 kalsa da aktivasyonun BF16/FP16 dtype'ini koru.
        return y.to(dtype) * self.weight.to(dtype=dtype)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, theta: float):
        super().__init__()
        assert head_dim % 2 == 0
        inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        pos = torch.arange(max_seq_len).float()
        freq = torch.outer(pos, inv)
        self.register_buffer("cos", freq.cos(), persistent=False)
        self.register_buffer("sin", freq.sin(), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, H, S, D]
        seq = x.shape[-2]
        cos = self.cos[:seq].to(device=x.device, dtype=x.dtype)[None, None, :, :]
        sin = self.sin[:seq].to(device=x.device, dtype=x.dtype)[None, None, :, :]
        even, odd = x[..., 0::2], x[..., 1::2]
        out = torch.empty_like(x)
        out[..., 0::2] = even * cos - odd * sin
        out[..., 1::2] = even * sin + odd * cos
        return out


class GQAttention(nn.Module):
    def __init__(self, cfg: ANKAConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.n_heads % cfg.n_kv_heads == 0
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.dropout = cfg.dropout
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.n_heads * self.head_dim, cfg.d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, cfg.max_seq_len, cfg.rope_theta)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, s, _ = x.shape
        q = self.q_proj(x).view(b, s, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, s, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, s, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = self.rope(q), self.rope(k)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
            enable_gqa=(self.n_heads != self.n_kv_heads),
        )
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, s, -1))


class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down(F.silu(self.gate(x)) * self.up(x))


def causal_lm_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    return F.cross_entropy(
        logits[:, :-1].contiguous().view(-1, logits.size(-1)),
        labels[:, 1:].contiguous().view(-1),
    )


def initialize_model(model: nn.Module, std: float = 0.02) -> None:
    for module in model.modules():
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=std)

## 3. Define the R3 Reflective Loop architecture

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 3 — R3: ANKA Reflective Loop
class ReflectiveBlock(nn.Module):
    def __init__(self, cfg: ANKAConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff)

    def forward(
        self,
        x: torch.Tensor,
        reflection: Optional[torch.Tensor],
        reflection_gate: Optional[torch.Tensor],
        residual_scale: float,
    ) -> torch.Tensor:
        conditioned = x if reflection is None else x + reflection_gate * reflection
        x = x + residual_scale * self.attn(self.attn_norm(conditioned))
        conditioned = x if reflection is None else x + reflection_gate * reflection
        x = x + residual_scale * self.ffn(self.ffn_norm(conditioned))
        return x


class ANKAReflectiveLoop(nn.Module):
    # 13 physical blocks, two shared passes, and discrepancy-conditioned reflection.
    def __init__(self, cfg: ANKAConfig, draft_weight: float = 0.25, aux_stride: int = 8):
        super().__init__()
        self.config = cfg
        self.draft_weight = draft_weight
        self.aux_stride = aux_stride
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([ReflectiveBlock(cfg) for _ in range(cfg.n_layers)])
        self.round_embedding = nn.Parameter(torch.empty(2, cfg.d_model))
        self.reflect_in_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_delta_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_proj = nn.Linear(2 * cfg.d_model, cfg.d_model, bias=False)
        self.reflect_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_gate_logits = nn.Parameter(torch.full((cfg.n_layers,), -2.1972246))
        self.draft_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.residual_scale = 1.0 / math.sqrt(2.0 * cfg.n_layers)
        self.last_metrics = {}

        initialize_model(self)
        nn.init.normal_(self.round_embedding, std=0.002)
        nn.init.normal_(self.reflect_proj.weight, std=0.01)

    def num_trainable_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None) -> ANKAOutput:
        if input_ids.shape[1] > self.config.max_seq_len:
            raise ValueError("Sequence exceeds max_seq_len.")

        token_state = self.embedding(input_ids)
        h = token_state + self.round_embedding[0]
        for block in self.blocks:
            h = block(h, None, None, self.residual_scale)
        draft = h

        discrepancy = self.reflect_delta_norm(draft - token_state)
        reflection = torch.tanh(self.reflect_proj(torch.cat([
            self.reflect_in_norm(draft), discrepancy
        ], dim=-1)))
        reflection = self.reflect_norm(reflection)

        h = draft + self.round_embedding[1]
        gates = torch.sigmoid(self.reflect_gate_logits)
        for layer_id, block in enumerate(self.blocks):
            h = block(h, reflection, gates[layer_id], self.residual_scale)

        logits = F.linear(self.final_norm(h), self.embedding.weight)
        loss = None
        final_loss = None
        draft_loss = None
        if labels is not None:
            final_loss = causal_lm_loss(logits, labels)
            if self.training and self.draft_weight > 0:
                # Use an auxiliary loss every eighth position instead of storing a full second logits tensor.
                draft_hidden = self.draft_norm(draft)[:, :-1:self.aux_stride]
                draft_targets = labels[:, 1::self.aux_stride]
                draft_logits = F.linear(draft_hidden, self.embedding.weight)
                draft_loss = F.cross_entropy(
                    draft_logits.reshape(-1, draft_logits.size(-1)),
                    draft_targets.reshape(-1),
                )
                loss = final_loss + self.draft_weight * draft_loss
            else:
                loss = final_loss

        self.last_metrics = {
            "final_loss": None if final_loss is None else final_loss.detach(),
            "draft_loss": None if draft_loss is None else draft_loss.detach(),
            "reflect_gate": gates.mean().detach(),
            "reflection_rms": reflection.float().square().mean().sqrt().detach(),
        }
        return ANKAOutput(logits=logits, loss=loss)


def build_r3() -> ANKAReflectiveLoop:
    cfg = ANKAConfig(
        name="R3",
        d_model=512,
        n_layers=13,       # 13 physical blocks, 26 effective block applications
        n_heads=8,
        n_kv_heads=2,
        d_ff=1536,
    )
    return ANKAReflectiveLoop(cfg)

## 4. Define the M3 Sparse Morphing-FFN architecture

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 4 — M3: ANKA Sparse Morphing-FFN
class SparseMorphFFN(nn.Module):
    # True top-1 token dispatch; unselected experts are not evaluated.
    def __init__(self, d_model: int, n_experts: int):
        super().__init__()
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.n_experts = n_experts
        self.last_balance = None
        self.last_entropy = None
        self.last_load = None

    def forward(self, x: torch.Tensor, experts: nn.ModuleList) -> torch.Tensor:
        shape = x.shape
        flat = x.reshape(-1, shape[-1])
        router_logits = self.router(flat).float()
        probs = torch.softmax(router_logits, dim=-1)
        choice = probs.argmax(dim=-1)
        out = torch.zeros_like(flat)

        for expert_id, expert in enumerate(experts):
            indices = torch.nonzero(choice == expert_id, as_tuple=False).flatten()
            if indices.numel() == 0:
                continue
            selected = flat.index_select(0, indices)
            expert_out = expert(selected)
            p = probs.index_select(0, indices)[:, expert_id].to(expert_out.dtype)
            straight_through = 1.0 + p - p.detach()
            dispatched = expert_out * straight_through.unsqueeze(-1)
            out.index_copy_(0, indices, dispatched.to(dtype=out.dtype))

        importance = probs.mean(dim=0)
        load = F.one_hot(choice, self.n_experts).float().mean(dim=0)
        self.last_balance = self.n_experts * torch.sum(importance * load)
        self.last_entropy = -(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()
        self.last_load = load.detach()
        return out.view(shape)


class MorphBlock(nn.Module):
    def __init__(self, cfg: ANKAConfig, stage_experts: nn.ModuleList):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.morph = SparseMorphFFN(cfg.d_model, len(stage_experts))
        self.residual_scale = 1.0 / math.sqrt(2.0 * cfg.n_layers)

    def forward(self, x: torch.Tensor, stage_experts: nn.ModuleList) -> torch.Tensor:
        x = x + self.residual_scale * self.attn(self.attn_norm(x))
        x = x + self.residual_scale * self.morph(self.ffn_norm(x), stage_experts)
        return x


class ANKASparseMorph(nn.Module):
    # 5 stage × 4 expert; each in-stage 4 layer same expert bank share.
    def __init__(self, cfg: ANKAConfig, n_stages: int = 5, experts_per_stage: int = 4):
        super().__init__()
        assert cfg.n_layers % n_stages == 0
        self.config = cfg
        self.n_stages = n_stages
        self.experts_per_stage = experts_per_stage
        self.layers_per_stage = cfg.n_layers // n_stages
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)

        self.stage_experts = nn.ModuleList([
            nn.ModuleList([
                SwiGLU(cfg.d_model, cfg.d_ff)
                for _ in range(experts_per_stage)
            ])
            for _ in range(n_stages)
        ])
        self.blocks = nn.ModuleList([
            MorphBlock(cfg, self.stage_experts[layer_id // self.layers_per_stage])
            for layer_id in range(cfg.n_layers)
        ])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.balance_weight = 0.01
        self.last_metrics = {}
        initialize_model(self)

    def num_trainable_parameters(self) -> int:
        # shared expert nesneleri only a kez sayilir.
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None) -> ANKAOutput:
        if input_ids.shape[1] > self.config.max_seq_len:
            raise ValueError("Sequence exceeds max_seq_len.")
        h = self.embedding(input_ids)
        balances, entropies = [], []
        stage_loads = [[] for _ in range(self.n_stages)]
        for layer_id, block in enumerate(self.blocks):
            stage_id = layer_id // self.layers_per_stage
            h = block(h, self.stage_experts[stage_id])
            balances.append(block.morph.last_balance)
            entropies.append(block.morph.last_entropy)
            stage_loads[stage_id].append(block.morph.last_load)

        logits = F.linear(self.final_norm(h), self.embedding.weight)
        balance_loss = torch.stack(balances).mean()
        lm_loss = causal_lm_loss(logits, labels) if labels is not None else None
        loss = None if lm_loss is None else lm_loss + self.balance_weight * balance_loss
        mean_loads = [torch.stack(items).mean(0) for items in stage_loads]
        self.last_metrics = {
            "lm_loss": None if lm_loss is None else lm_loss.detach(),
            "balance_loss": balance_loss.detach(),
            "router_entropy": torch.stack(entropies).mean().detach(),
            "stage_loads": torch.stack(mean_loads).detach(),
        }
        return ANKAOutput(logits=logits, loss=loss)


def build_m3() -> ANKASparseMorph:
    cfg = ANKAConfig(
        name="M3",
        d_model=448,
        n_layers=20,
        n_heads=7,
        n_kv_heads=1,
        d_ff=1216,
    )
    return ANKASparseMorph(cfg, n_stages=5, experts_per_stage=4)

## 5. Run structural, gradient, and causal tests

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 5 — R3 / M3 structural, gradient and causal tests

def structural_test(name: str, builder) -> dict:
    print("\n" + "=" * 74)
    print(name, "STRUCTURAL TEST")
    print("=" * 74)

    torch.cuda.empty_cache()

    model = builder().to(DEVICE)
    params = model.num_trainable_parameters()

    assert params < 50_000_000, (
        f"{name}: 50M parameter limit exceeded: {params:,}"
    )

    ids = torch.randint(
        0,
        model.config.vocab_size,
        (2, 24),
        device=DEVICE,
    )

    model.train()

    with torch.autocast(
        "cuda",
        dtype=torch.bfloat16,
    ):
        out = model(
            ids,
            labels=ids,
        )

    assert out.logits.shape == (
        2,
        24,
        model.config.vocab_size,
    )

    assert torch.isfinite(out.loss)
    out.loss.backward()

    important = [
        (n, p)
        for n, p in model.named_parameters()
        if p.requires_grad
        and (
            "reflect_proj" in n
            or "router" in n
        )
    ]

    assert important, (
        f"{name}: mechanism parameter not found"
    )

    missing = [
        n
        for n, p in important
        if p.grad is None
        or not torch.isfinite(p.grad).all()
    ]

    assert not missing, (
        f"{name}: gradient gelmeyen parametreler: "
        f"{missing[:5]}"
    )

    # Change future tokens.
    # Measure the logits difference over the first 11 positions.
    model.eval()

    a = torch.randint(
        0,
        model.config.vocab_size,
        (1, 20),
        device=DEVICE,
    )

    b = a.clone()

    b[:, 11:] = torch.randint(
        0,
        model.config.vocab_size,
        b[:, 11:].shape,
        device=DEVICE,
    )

    with (
        torch.inference_mode(),
        torch.autocast(
            "cuda",
            dtype=torch.bfloat16,
        ),
    ):
        ya = model(a).logits[:, :11].float()
        yb = model(b).logits[:, :11].float()

    causal_error = (
        ya - yb
    ).abs().max().item()

    # R3 for strict threshold.
    # M3 for sparse BF16 dispatch caused-by
    # numerical difference tolerating that temporary threshold.
    if name == "R3":
        causal_tolerance = 0.002
    else:
        causal_tolerance = 0.25

    print(
        "Causal error           :",
        causal_error,
    )

    print(
        "used causal threshold:",
        causal_tolerance,
    )

    assert causal_error < causal_tolerance, (
        f"{name}: causal fark limit was-exceeded. "
        f"error={causal_error}, "
        f"threshold={causal_tolerance}"
    )

    if name == "R3":
        assert len(model.blocks) == 13

        print(
            "physical/efektif blok: 13 / 26"
        )

        print(
            "Reflect gate         :",
            float(
                model.last_metrics[
                    "reflect_gate"
                ]
            ),
        )

    else:
        assert sum(
            len(stage)
            for stage in model.stage_experts
        ) == 20

        print(
            "expert bankasi        : "
            "5 stage × 4 expert"
        )

        print(
            "Stage loads        :\n",
            model.last_metrics[
                "stage_loads"
            ].cpu().numpy(),
        )

    result = {
        "arm": name,
        "parameters": params,
        "causal_error": causal_error,
        "causal_tolerance": causal_tolerance,
    }

    print(
        "parameter             :",
        f"{params:,}",
    )

    print(
        "Loss                  :",
        f"{out.loss.item():.4f}",
    )

    print(
        "RESULT                 : PASS"
    )

    del model
    del out
    del ids
    del a
    del b
    del ya
    del yb

    gc.collect()
    torch.cuda.empty_cache()

    return result


test_results = [
    structural_test(
        "R3",
        build_r3,
    ),
    structural_test(
        "M3",
        build_m3,
    ),
]

print(
    "\nALL STRUCTURAL TESTS PASSED:",
    test_results,
)

## 6. Discover binary shards and build the token DataLoader

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 6 — Binary shard discovery and token DataLoader

def collect_paths(obj) -> list[str]:
    found = []

    if isinstance(obj, str):
        if obj.endswith((".bin", ".npy")):
            found.append(obj)

    elif isinstance(obj, dict):
        for value in obj.values():
            found.extend(
                collect_paths(value)
            )

    elif isinstance(obj, list):
        for value in obj:
            found.extend(
                collect_paths(value)
            )

    return found


def resolve_shards(
    manifest_path: Path,
) -> list[Path]:

    manifest = json.loads(
        manifest_path.read_text()
    )

    raw_paths = collect_paths(
        manifest
    )

    paths = []

    for item in raw_paths:
        path = Path(item)

        candidates = [
            path,
            manifest_path.parent / path,
            manifest_path.parent / path.name,
        ]

        resolved = next(
            (
                candidate.resolve()
                for candidate in candidates
                if candidate.exists()
            ),
            None,
        )

        if (
            resolved is not None
            and resolved not in paths
        ):
            paths.append(resolved)

    if not paths:
        paths = sorted(
            manifest_path.parent.glob("*.bin")
        )

    if len(paths) < 2:
        raise RuntimeError(
            "most few two shard required. "
            f"found shardlar: {paths}"
        )

    return paths


SHARDS = resolve_shards(
    MANIFEST_PATH
)

TRAIN_SHARDS = SHARDS[:-1]
VAL_SHARD = SHARDS[-1]

print(
    "Train shard count:",
    len(TRAIN_SHARDS),
)

print(
    "Validation shard:",
    VAL_SHARD,
)


class RandomTokenDataset(
    IterableDataset
):

    def __init__(
        self,
        shards: list[Path],
        seq_len: int,
        seed: int,
    ):
        super().__init__()

        self.shards = shards
        self.seq_len = seq_len
        self.seed = seed

    def __iter__(self):

        worker_info = (
            torch.utils.data
            .get_worker_info()
        )

        worker_id = (
            0
            if worker_info is None
            else worker_info.id
        )

        rng = np.random.default_rng(
            self.seed
            + 10_003 * worker_id
        )

        arrays = []

        for path in self.shards:

            if path.suffix == ".npy":
                array = np.load(
                    path,
                    mmap_mode="r",
                )
            else:
                array = np.memmap(
                    path,
                    mode="r",
                    dtype=np.uint16,
                )

            arrays.append(array)

        sizes = np.asarray(
            [
                max(
                    0,
                    len(array)
                    - self.seq_len
                    - 1,
                )
                for array in arrays
            ],
            dtype=np.float64,
        )

        if sizes.sum() <= 0:
            raise RuntimeError(
                "Sequences will be generated from shards "
                "how-much token unavailable."
            )

        probabilities = (
            sizes / sizes.sum()
        )

        while True:

            shard_id = int(
                rng.choice(
                    len(arrays),
                    p=probabilities,
                )
            )

            array = arrays[shard_id]

            max_start = (
                len(array)
                - self.seq_len
                - 1
            )

            start = int(
                rng.integers(
                    0,
                    max_start + 1,
                )
            )

            tokens = np.asarray(
                array[
                    start:
                    start + self.seq_len
                ],
                dtype=np.int64,
            ).copy()

            yield torch.from_numpy(
                tokens
            )


def make_loader(
    seq_len: int,
    batch_size: int,
    seed: int,
) -> DataLoader:

    dataset = RandomTokenDataset(
        shards=TRAIN_SHARDS,
        seq_len=seq_len,
        seed=seed,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )


def fixed_validation_batches(
    seq_len: int,
    batch_size: int,
    n_batches: int = 40,
):

    if VAL_SHARD.suffix == ".npy":
        array = np.load(
            VAL_SHARD,
            mmap_mode="r",
        )
    else:
        array = np.memmap(
            VAL_SHARD,
            mode="r",
            dtype=np.uint16,
        )

    required = (
        batch_size
        * n_batches
    )

    max_start = (
        len(array)
        - seq_len
        - 1
    )

    starts = np.linspace(
        0,
        max_start,
        required,
        dtype=np.int64,
    )

    batches = []

    for batch_id in range(n_batches):

        begin = (
            batch_id
            * batch_size
        )

        end = begin + batch_size

        chunk = starts[begin:end]

        rows = []

        for start in chunk:

            row = np.asarray(
                array[
                    start:
                    start + seq_len
                ],
                dtype=np.int64,
            ).copy()

            rows.append(row)

        batch = torch.from_numpy(
            np.stack(rows)
        )

        batches.append(batch)

    return batches

## 7. Configure the training engine and A100 settings

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 7 — Training settings and motoru

TOKEN_MULTIPLIER = 20.0
SEQ_LEN = 512

# A100-80GB target ayari.
# If OOM occurs, reduce only the R3 batch value to 96 or 64.
MICRO_BATCH = {
    "R3": 64,
    "M3": 64,
}

GRAD_ACCUM = {
    "R3": 1,
    "M3": 1,
}

PEAK_LR = {
    "R3": 2.5e-3,
    "M3": 2.5e-3,
}

MIN_LR_RATIO = 0.10
WEIGHT_DECAY = 0.10
WARMUP_RATIO = 0.02

LOG_EVERY = 10
MEASURE_WARMUP_STEPS = 5
VAL_EVERY = 500
SAVE_EVERY = 500
VAL_BATCHES = 20


def lr_at(
    step: int,
    total_steps: int,
    peak: float,
) -> float:
    warmup = max(
        1,
        int(total_steps * WARMUP_RATIO),
    )

    if step < warmup:
        return peak * (step + 1) / warmup

    progress = (
        step - warmup
    ) / max(
        1,
        total_steps - warmup,
    )

    progress = min(progress, 1.0)

    cosine = 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

    return peak * (
        MIN_LR_RATIO
        + (1.0 - MIN_LR_RATIO) * cosine
    )


@torch.inference_mode()
def evaluate(
    model: nn.Module,
    batches,
) -> float:
    model.eval()

    losses = []

    for cpu_batch in batches:
        batch = cpu_batch.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            "cuda",
            dtype=torch.bfloat16,
        ):
            output = model(
                batch,
                labels=batch,
            )

        metric = model.last_metrics.get(
            "final_loss",
            model.last_metrics.get(
                "lm_loss",
                output.loss,
            ),
        )

        losses.append(
            float(metric)
        )

    model.train()

    return sum(losses) / len(losses)


def save_checkpoint(
    path: Path,
    model,
    optimizer,
    step: int,
    seen_tokens: int,
    history: list,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = path.with_suffix(
        ".tmp"
    )

    torch.save(
        {
            "arm": model.config.name,
            "model_config": asdict(
                model.config
            ),
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "step": step,
            "seen_tokens": seen_tokens,
            "history": history,
        },
        temp_path,
    )

    temp_path.replace(path)


def train_arm(
    arm: str,
    builder,
) -> Path:

    assert arm in (
        "R3",
        "M3",
    )

    output_dir = OUTPUT_ROOT / arm
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    latest_path = output_dir / "latest.pt"
    final_path = output_dir / "final.pt"

    # Do not restart a completed training run.
    if final_path.exists():
        print(
            f"{arm}: final checkpoint zaten var."
        )
        print(
            "Training was not restarted:"
        )
        print(final_path)
        return final_path

    model = builder().to(DEVICE)

    parameter_count = (
        model.num_trainable_parameters()
    )

    token_budget = int(
        parameter_count * TOKEN_MULTIPLIER
    )

    batch_size = MICRO_BATCH[arm]
    accumulation = GRAD_ACCUM[arm]

    tokens_per_step = (
        batch_size
        * (SEQ_LEN - 1)
        * accumulation
    )

    total_steps = math.ceil(
        token_budget / tokens_per_step
    )

    validation_batches = (
        fixed_validation_batches(
            SEQ_LEN,
            8,
            VAL_BATCHES,
        )
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PEAK_LR[arm],
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=WEIGHT_DECAY,
        fused=True,
    )

    start_step = 0
    seen_tokens = 0
    history = []

    if (
        latest_path.exists()
        and not final_path.exists()
    ):
        checkpoint = torch.load(
            latest_path,
            map_location="cpu",
            weights_only=False,
        )

        model.load_state_dict(
            checkpoint["model"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer"]
        )

        start_step = int(
            checkpoint["step"]
        )

        seen_tokens = int(
            checkpoint["seen_tokens"]
        )

        history = checkpoint.get(
            "history",
            [],
        )

        print(
            f"{arm}: step {start_step}"
            " is being resumed."
        )

    loader_seed = (
        SEED
        + start_step * 1_000_003
    )

    loader = make_loader(
        SEQ_LEN,
        batch_size,
        loader_seed,
    )

    iterator = iter(loader)

    print("\n" + "=" * 78)
    print(f"{arm} long training")
    print("=" * 78)

    print(
        "parameter       :",
        f"{parameter_count:,}",
    )

    print(
        "Token budget   :",
        f"{token_budget:,}",
    )

    print(
        "Step            :",
        f"{total_steps:,}",
    )

    print(
        "Micro batch     :",
        batch_size,
    )

    print(
        "Gradient accum  :",
        accumulation,
    )

    print(
        "Token/step      :",
        f"{tokens_per_step:,}",
    )

    model.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    torch.cuda.reset_peak_memory_stats()

    interval_start = time.perf_counter()
    interval_tokens = 0
    interval_losses = []
    interval_data_wait = 0.0
    interval_steps = 0

    for step in range(
        start_step,
        total_steps,
    ):

        current_lr = lr_at(
            step,
            total_steps,
            PEAK_LR[arm],
        )

        for group in optimizer.param_groups:
            group["lr"] = current_lr

        for _ in range(accumulation):

            data_start = time.perf_counter()

            batch = next(
                iterator
            ).to(
                DEVICE,
                non_blocking=True,
            )

            interval_data_wait += (
                time.perf_counter()
                - data_start
            )

            with torch.autocast(
                "cuda",
                dtype=torch.bfloat16,
            ):
                output = model(
                    batch,
                    labels=batch,
                )

                loss = (
                    output.loss
                    / accumulation
                )

            loss.backward()

            interval_losses.append(
                output.loss.detach()
            )

            interval_tokens += (
                batch_size
                * (SEQ_LEN - 1)
            )

            seen_tokens += (
                batch_size
                * (SEQ_LEN - 1)
            )

        grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )
        )

        optimizer.step()

        optimizer.zero_grad(
            set_to_none=True
        )

        completed_step = step + 1
        interval_steps += 1

        # Exclude first-worker and memmap initialization from timing.
        if (
            completed_step
            == MEASURE_WARMUP_STEPS
        ):
            torch.cuda.synchronize()

            interval_start = (
                time.perf_counter()
            )

            interval_tokens = 0
            interval_losses = []
            interval_data_wait = 0.0
            interval_steps = 0

        if (
            completed_step % LOG_EVERY
            == 0
        ):
            torch.cuda.synchronize()

            elapsed = (
                time.perf_counter()
                - interval_start
            )

            tokens_per_second = (
                interval_tokens
                / max(elapsed, 1e-9)
            )

            average_loss = float(
                torch.stack(
                    interval_losses
                ).mean()
            )

            data_wait_ms = (
                1000.0
                * interval_data_wait
                / max(interval_steps, 1)
            )

            record = {
                "step": completed_step,
                "tokens": seen_tokens,
                "loss": average_loss,
                "lr": current_lr,
                "grad_norm": float(
                    grad_norm
                ),
                "tokens_per_second": (
                    tokens_per_second
                ),
                "data_wait_ms_per_step": (
                    data_wait_ms
                ),
            }

            history.append(record)

            print(
                f"[{arm}] "
                f"step={completed_step:05d}/"
                f"{total_steps} "
                f"loss={average_loss:.4f} "
                f"lr={current_lr:.3e} "
                f"grad={float(grad_norm):.3f} "
                f"tok/s={tokens_per_second:,.0f} "
                f"data_wait={data_wait_ms:.1f}ms/step "
                f"peak="
                f"{torch.cuda.max_memory_allocated() / 2**30:.2f}"
                "GiB"
            )

            interval_start = (
                time.perf_counter()
            )

            interval_tokens = 0
            interval_losses = []
            interval_data_wait = 0.0
            interval_steps = 0

        if (
            completed_step % VAL_EVERY
            == 0
            or completed_step == total_steps
        ):
            validation_loss = evaluate(
                model,
                validation_batches,
            )

            perplexity = math.exp(
                min(validation_loss, 20)
            )

            print(
                f"[{arm}] "
                f"validation_loss="
                f"{validation_loss:.6f} "
                f"ppl={perplexity:.2f}"
            )

        if (
            completed_step % SAVE_EVERY
            == 0
            or completed_step == total_steps
        ):
            save_checkpoint(
                latest_path,
                model,
                optimizer,
                completed_step,
                seen_tokens,
                history,
            )

            torch.cuda.synchronize()
            interval_start = (
                time.perf_counter()
            )

    save_checkpoint(
        final_path,
        model,
        optimizer,
        total_steps,
        seen_tokens,
        history,
    )

    final_validation_loss = evaluate(
        model,
        validation_batches,
    )

    summary = {
        "arm": arm,
        "parameters": parameter_count,
        "token_budget": token_budget,
        "seen_tokens": seen_tokens,
        "steps": total_steps,
        "final_validation_loss": (
            final_validation_loss
        ),
        "peak_vram_gib": (
            torch.cuda.max_memory_allocated()
            / 2**30
        ),
    }

    summary_path = (
        output_dir / "summary.json"
    )

    summary_path.write_text(
        json.dumps(
            summary,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        json.dumps(
            summary,
            indent=2,
        )
    )

    del model
    del optimizer
    del loader
    del iterator

    gc.collect()
    torch.cuda.empty_cache()

    return final_path

## 8. Clear RAM and VRAM before each arm

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# RAM / VRAM TEMIZLIGI
import gc
import torch

def clear_training_memory():
    global_names = [
        "model",
        "optimizer",
        "loader",
        "iterator",
        "R3_CHECKPOINT",
        "M3_CHECKPOINT",
        "checkpoint",
        "output",
        "loss",
        "batch",
    ]

    for name in global_names:
        obj = globals().pop(name, None)

        if obj is None:
            continue

        try:
            if hasattr(obj, "_shutdown_workers"):
                obj._shutdown_workers()
        except Exception:
            pass

        try:
            del obj
        except Exception:
            pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

    print("RAM and VRAM temizlendi.")
    print(
        f"VRAM allocated: "
        f"{torch.cuda.memory_allocated() / 2**30:.2f} GiB"
    )
    print(
        f"VRAM reserved : "
        f"{torch.cuda.memory_reserved() / 2**30:.2f} GiB"
    )


clear_training_memory()

## 9. Train R3 to the target token budget

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
R3_CHECKPOINT = train_arm(
    "R3",
    build_r3,
)

## 10. Train M3 to the target token budget

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 9 — M3 long training
# R3 fully after-completion after run.
M3_CHECKPOINT = train_arm("M3", build_m3)

## 11. Compare fixed validation metrics and generations

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 10 — Joint comparison with fixed validation and sample generation
!pip -q install tokenizers
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
EOS_ID = tokenizer.token_to_id("<eos>")


@torch.inference_mode()
def generate(model, prompt: str, max_new_tokens: int = 80, temperature: float = 0.8, top_k: int = 40):
    ids = tokenizer.encode(prompt).ids
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    model.eval()
    for _ in range(max_new_tokens):
        context = x[:, -model.config.max_seq_len:]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(context).logits[:, -1].float()
        logits /= temperature
        if top_k:
            values, indices = torch.topk(logits, min(top_k, logits.shape[-1]))
            filtered = torch.full_like(logits, -float("inf"))
            filtered.scatter_(1, indices, values)
            logits = filtered
        next_id = torch.multinomial(torch.softmax(logits, -1), 1)
        x = torch.cat([x, next_id], dim=1)
        if EOS_ID is not None and int(next_id) == EOS_ID:
            break
    return tokenizer.decode(x[0].tolist(), skip_special_tokens=True)


def load_final(arm: str):
    builder = {"R3": build_r3, "M3": build_m3}[arm]
    path = OUTPUT_ROOT / arm / "final.pt"
    assert path.exists(), f"missing checkpoint: {path}"
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    model = builder()
    model.load_state_dict(checkpoint["model"])
    return model.to(DEVICE), checkpoint


prompts = [
    "The future of",
    "Science is important because",
    "If a box is placed on a table",
    "One day in Istanbul",
]
comparison_batches = fixed_validation_batches(SEQ_LEN, 8, 100)
reports = []

for arm in ("R3", "M3"):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model, checkpoint = load_final(arm)
    start = time.perf_counter()
    loss = evaluate(model, comparison_batches)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    evaluated_tokens = len(comparison_batches) * 8 * (SEQ_LEN - 1)
    samples = [{"prompt": p, "completion": generate(model, p)} for p in prompts]
    report = {
        "arm": arm,
        "parameters": model.num_trainable_parameters(),
        "seen_tokens": checkpoint["seen_tokens"],
        "validation_loss": loss,
        "perplexity": math.exp(min(loss, 20)),
        "eval_tokens_per_second": evaluated_tokens / elapsed,
        "peak_vram_gib": torch.cuda.max_memory_allocated() / 2**30,
        "samples": samples,
    }
    if arm == "R3":
        report["reflect_gate"] = float(model.last_metrics["reflect_gate"])
    else:
        report["router_entropy"] = float(model.last_metrics["router_entropy"])
        report["stage_loads"] = model.last_metrics["stage_loads"].cpu().tolist()
    reports.append(report)
    print("\n", json.dumps(report, indent=2, ensure_ascii=False))
    del model, checkpoint
    gc.collect()
    torch.cuda.empty_cache()

reports.sort(key=lambda x: x["validation_loss"])
report_path = OUTPUT_ROOT / "r3_m3_comparison.json"
report_path.write_text(json.dumps(reports, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nSiralama:")
for rank, row in enumerate(reports, 1):
    print(
        f"{rank}. {row['arm']} | loss={row['validation_loss']:.6f} | "
        f"ppl={row['perplexity']:.2f} | eval={row['eval_tokens_per_second']:,.0f} tok/s"
    )
print("report:", report_path)